# Co-visitation baseline

Issue: #18

## Question

How far can we get by recommending items that were viewed, carted or ordered together with the items in the session, with no learning at all? On OTTO this is a very strong bar, so it is the honest one the learned models must clear.

## Data used

Inputs: `data/covis/*` (#7, training window w0 only), `data/categories/item_category.parquet` (#8), `data/eval/ranking_queries.parquet` (#10), `data/results/popularity.parquet` (#17, for the comparison).

Output: `data/results/covis.parquet` and `data/results/covis_pure.parquet`. Code: `notebooks/covis_rank.py`, `evalset.py` (the shared rules from #17: rank inside the query category, no prefix items, the same 56,544 queries).

## How it ranks

For a session prefix, each of the last 30 events adds its co-visitation row to a score, multiplied by `type weight * 0.8 ** (position from the end)`. Type weights are click 1, cart 3, order 6, so carts and orders and recent events count more. Candidates are then restricted to the query category, prefix items are removed, and the top 100 are returned.

Two versions are reported:
- **covis (pure)**: only items with co-visitation evidence. The list can be shorter than 100.
- **covis + popularity fill** (the main one): if fewer than 100 items remain, the list is topped up with the most popular items of the category (same popularity as #17). Prefixes with no neighbours at all get the plain popularity list. How often this happens is counted.

In [ ]:
import resource
import sys
from pathlib import Path

import numpy as np
import polars as pl
from scipy import sparse

sys.path.insert(0, ".")
sys.path.insert(0, "notebooks")
from bootstrap import paired_bootstrap_diff
from covis_rank import CovisRanker
from evalset import TOP, evaluate, prepare_eval, summarize
from split import WINDOWS, day_to_ms

DATA = Path("data") if Path("data").exists() else Path("../data")
TRAIN_END = day_to_ms(WINDOWS["w0 initial (weeks 1-2)"]["train_end_day"])
OUT = DATA / "results"

## A toy check

Items 1 to 4. Item 1 is covisited with 2 (score 5), 3 (score 1) and 4 (score 9). Categories: items 2 and 3 are in category 0, item 4 in category 1. Item 5 has no neighbours.

In [ ]:
toy_m = sparse.csr_matrix(np.array([
    [0, 5, 1, 9, 0],
    [5, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [9, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
], dtype=np.float32))
toy_ids = np.array([1, 2, 3, 4, 5])
toy_cat = {1: 0, 2: 0, 3: 0, 4: 1, 5: 0}
fill = {0: np.array([3, 2, 1, 5]), 1: np.array([4])}
r = CovisRanker(toy_m, toy_ids, toy_cat, fill_by_cat=fill, top=3)

q = {"prefix_list": [1], "prefix_types": [0], "prefix": {1}, "category": 0}
assert r.rank(q, pure=True) == [2, 3]          # item 4 is in another category; 2 (5) before 3 (1)
assert r.rank(q) == [2, 3, 5]                  # padded from the popularity list, skipping used items and the prefix
q2 = {"prefix_list": [1, 2], "prefix_types": [0, 0], "prefix": {1, 2}, "category": 0}
assert r.rank(q2, pure=True) == [3]            # prefix item 2 is not returned; 3 remains
q3 = {"prefix_list": [5], "prefix_types": [0], "prefix": {5}, "category": 0}
assert r.rank(q3, pure=True) == []             # no neighbours
assert r.rank(q3) == [3, 2, 1]                 # fully the popularity fallback (prefix item excluded)
# type weights: a cart of item 1 counts 3x a click, so with two prefix items the higher-weighted one dominates
toy2 = sparse.csr_matrix(np.array([[0, 0, 1, 0], [0, 0, 0, 1], [1, 0, 0, 0], [0, 1, 0, 0]], dtype=np.float32))
r2 = CovisRanker(toy2, np.array([1, 2, 3, 4]), {1: 0, 2: 0, 3: 0, 4: 0}, top=2, decay=1.0)
qq = {"prefix_list": [1, 2], "prefix_types": [0, 1], "prefix": {1, 2}, "category": 0}   # click item 1, cart item 2
assert r2.rank(qq, pure=True) == [4, 3]        # item 4 (from the cart, weight 3) beats item 3 (from the click, weight 1)
print("toy checks pass")

## Build the ranker on the training-window matrix

In [ ]:
matrix = sparse.load_npz(DATA / "covis" / "covis.npz")
item_ids = np.load(DATA / "covis" / "item_ids.npy")
import json

meta = json.loads((DATA / "covis" / "meta.json").read_text())
assert meta["train_end_ms"] == TRAIN_END, "matrix is not from the training window"

categories = pl.read_parquet(DATA / "categories" / "item_category.parquet")
item_cluster = dict(zip(categories["aid"].to_list(), categories["cluster"].to_list()))

events = pl.scan_parquet(DATA / "sample" / "events.parquet")
pop = events.filter((pl.col("type") == 0) & (pl.col("ts") < TRAIN_END)).group_by("aid").agg(pl.len().alias("clicks")).collect()
ranked = (categories.join(pop, on="aid", how="left").with_columns(pl.col("clicks").fill_null(0))
          .sort(["cluster", "clicks", "aid"], descending=[False, True, False]))
fill_by_cat = {c: part["aid"].to_numpy()[:2000] for (c,), part in ranked.group_by("cluster", maintain_order=True)}

queries = pl.read_parquet(DATA / "eval" / "ranking_queries.parquet")
prepared, prep_stats = prepare_eval(queries)
print(prep_stats)

## Evaluate both versions

In [ ]:
ranker = CovisRanker(matrix, item_ids, item_cluster, fill_by_cat=fill_by_cat)
res = evaluate(prepared, ranker.rank)
counts_main = dict(ranker.counts)

pure_ranker = CovisRanker(matrix, item_ids, item_cluster, fill_by_cat=fill_by_cat)
res_pure = evaluate(prepared, lambda q: pure_ranker.rank(q, pure=True))

def show(name, results):
    s = summarize(results)
    print(name)
    for k, v in s.items():
        if isinstance(v, tuple):
            print(f"  {k:>22}: {v[0]:.4f}  [{v[1]:.4f}, {v[2]:.4f}]")
        else:
            print(f"  {k:>22}: {v:.4f}  (OTTO official aggregate)")
    return s

s_main = show("covis + popularity fill (main)", res)
s_pure = show("covis pure", res_pure)
n = res.height
print(f"\nqueries with no neighbours at all (popularity list used): {counts_main['no_neighbours']:,} ({counts_main['no_neighbours'] / n:.1%})")
print(f"queries where the list was topped up with popularity: {counts_main['filled']:,} ({counts_main['filled'] / n:.1%})")
print(f"pure version: lists shorter than 100: {(res_pure['n_returned'] < TOP).sum():,}, empty lists: {(res_pure['n_returned'] == 0).sum():,}")

## Does it beat the popularity baseline?

Paired bootstrap over sessions on the per-session difference (same sessions, same metric).

In [ ]:
pop_res = pl.read_parquet(OUT / "popularity.parquet")
joined = res.join(pop_res.select("session", pl.col("ndcg10").alias("pop_ndcg10"), pl.col("recall20").alias("pop_recall20")), on="session", how="inner")
assert joined.height == res.height, "the two methods must be scored on the same sessions"
for col, pop_col in (("ndcg10", "pop_ndcg10"), ("recall20", "pop_recall20")):
    mean, lo, hi = paired_bootstrap_diff(joined[col].to_numpy(), joined[pop_col].to_numpy())
    print(f"{col}: covis minus popularity = {mean:+.4f}  [{lo:+.4f}, {hi:+.4f}]")
    assert lo > 0, f"covis does not clearly beat popularity on {col}"
print("co-visitation beats popularity with an interval that excludes zero")

## Save

In [ ]:
res.write_parquet(OUT / "covis.parquet")
res_pure.write_parquet(OUT / "covis_pure.parquet")
print("saved covis.parquet and covis_pure.parquet;", res.height, "sessions")
print("peak memory GB:", round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e9, 2))

## Why this is the honest bar

Co-visitation needs no training beyond counting, yet it uses the session itself, which is the strongest signal in this data: what a user just looked at predicts what they look at next. A learned model that cannot beat it is not adding anything over counting. Popularity ignores the session, which is why it does so much worse.

## What I learned

To be written by the owner of the project.